# Table of Contents

1. Pulling the Data](
1. Preparing the Data
1. Further Cleaning

# Pulling the Data

All of the data collected from others was collected

In [251]:
# !pip install gspread pandas google-auth
import gspread
from google.oauth2.service_account import Credentials
import os, pandas as pd
from pathlib import Path


In [252]:
os.environ["GOOGLE_SHEETS_CREDENTIALS"] = (
    "/opt/notebooks/psalms_nlp_sp26/private/psalms-blind-scoring-da42a38adf3c.json"
)

os.getcwd()

'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [253]:
# Current notebook directory
notebook_dir = Path.cwd()

# Build path to the JSON
cred_path = notebook_dir.parent / "private" / "psalms-blind-scoring-da42a38adf3c.json"
print("Credential path exists?", cred_path.exists())

os.getcwd()

Credential path exists? True


'/opt/notebooks/psalms_nlp_sp26/query_compare'

In [254]:
import socket
socket.gethostbyname("oauth2.googleapis.com")


'172.253.115.95'

In [255]:
creds = Credentials.from_service_account_file(
    os.environ["GOOGLE_SHEETS_CREDENTIALS"],
    scopes=[
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]
)

client = gspread.authorize(creds)

sheet = client.open("results_scored")

In [256]:
# Access second sheet (index 1)
worksheet2 = sheet.get_worksheet(1)

# Get all values
data = worksheet2.get_all_values()

# Convert to DataFrame (first row as header)
df = pd.DataFrame(data[1:], columns=data[0])

# storing the collected data for reference later
df.to_csv("../data/results_scored.csv", index=False, mode="w")


In [257]:
df

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,5,p06,1,p03,3,p08
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,1,p03,8,p04,7,p06
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,7,p06,1,p03,3,p10
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,7,p06,6,p05,5,p09
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,2,p01,0,p03,2,p08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",2,0,p01,0,p06,7,p04
232,Verses where the psalmist remembers past deliv...,TFIDF,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,1,0,p03,0,p17,,
233,Verses where the psalmist remembers past deliv...,TFIDF,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,3,10,p06,,,,
234,Verses where the psalmist remembers past deliv...,TFIDF,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",8,2,p10,7,p05,,


# Preparing the data
I want each row to hold one of the four possible scored for each of the `236 results`. So if rebuilt properly, we should end up with a total of: 
$$236 * 4 = 944\ results$$

I need to start by unpivoting my own score separte from the other scores.

## Numbering the Results 
I also want to be able to reference the order of the results within each search. I collected the top 5 results from each search. There was a bug in my code that took the top 6 results from searches. I am going to just worry about the top 5 results to keep everything fair. 

In [258]:
# temporary dataframe to not break the original 
temp = df.copy()

# aqdding a column to number the indivudal results
temp['numbered_result'] = pd.NA
#temp

In [259]:
n = temp.shape[0]

# starting with the number 1 result of a query
num_result = 1

query = temp["Query"].iloc[0]
method = temp["Method"].iloc[0]

for i in range(n):
    # checking if we are in the same group fo data to be numbered
    if query == temp["Query"].iloc[i] and method == temp["Method"].iloc[i]:
        temp["numbered_result"].iloc[i] = num_result
        num_result += 1
        
    # in an new group of results
    else:
        # the current result is the number result for the new set of results
        temp["numbered_result"].iloc[i] = 1
        # reset the number result
        num_result = 2
        # update to the new target query & method
        query = temp["Query"].iloc[i]
        method = temp["Method"].iloc[i]
        
# temp.tail(20)


/tmp/ipykernel_248/2796122015.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  temp["numbered_result"].iloc[i] = num_result
/tmp/ipykernel_248/2796122015.py:12: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!


In [260]:
# select first 7 columns + last column
cols_to_keep = list(temp.columns[:7]) + [temp.columns[-1]]
caden = temp[cols_to_keep]

# caden.head()


In [261]:
caden['User'] = 'caden'

caden = caden.rename(columns={"CadenScore": "Score"})

# caden

/tmp/ipykernel_248/2631933811.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  caden['User'] = 'caden'


In [262]:
caden  = caden[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User', 'Score']]
caden

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",caden,2
232,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,caden,1
233,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,caden,3
234,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",caden,8


Moving on to prepaering the external scores.

In [263]:
external = temp[['Query', 'Method', "numbered_result", 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'User1', 'Score1', 'User2', 'Score2', 'User3', 'Score3']]

#external

In [264]:
# Unpivot User/Score pairs
df_long = pd.wide_to_long(
    external,
    stubnames=["User", "Score"],  # the base column names
    i=["Query", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", "Verse"],  # columns to keep
    j="Pair",  # new column for the pair number
    sep=""      # number comes directly after the stub name
).reset_index()

# Optional: reorder columns
df_long = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "Pair", "User", "Score"]]



In [265]:
external = df_long[["Query", "Method", "numbered_result",  "Similarity Score (%)", "Text", "Psalm Num", "Verse", "User", "Score"]]

external

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8
...,...,...,...,...,...,...,...,...,...
703,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7
704,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",,
705,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7
706,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0


In [266]:
#caden.head()

## Combining the Prepared Data back together

In [267]:
scores = pd.concat([caden, external], ignore_index=True)
scores

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
939,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",p05,7
940,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",,
941,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p01,7
942,Verses where the psalmist remembers past deliv...,TFIDF,6,5.46,Psalter,131,"Lord, remember David and all his meekness; how...",p06,0


In [268]:
(scores["Score"].notna() & (scores["Score"] != "")).sum()

848

In [269]:
#scores["Query"].unique()

We now have our intended 944 rows of data we can move on to do last few preperations for the analysis. 

# Further Data Cleaning <a href="further_cleaning"></a>

I now need to work on some of the anaiysis of the data and alot of the imediate processiong was done within another notebook so it will be coppied into here. 

## Query Categoization

In [270]:
query_categories = {
    "mercy": "Simple Keyword Queries",
    "prayer":"Simple Keyword Queries",
    "The Lord is my shepherd": "Phrase/Exact Match Queries",
    "Create in me a clean heart":"Phrase/Exact Match Queries",
    "protection from enemies": "Thematic/Semantic Queries",
    "praise in times of suffering": "Thematic/Semantic Queries",
    "How does the psalmist express trust in God while surrounded by fear and uncertainty?":
        "Long/Complex Queries",
    "Verses where the psalmist remembers past deliverance and uses it to find hope in present trials.":
        "Long/Complex Queries",
    "Rejoice, O ye heavens, sound the trumpets, ye foundation of the earth, thunder forth gladness, O ye mountains: for behold, Emanuel to the Cross our sins, and the Giver of Life hath slain death, raising up Adam; for He loveth mankind.":
        "Orthodox Service Quotes",
    "Have mercy on me, O God, have mercy on me. For my soul trusts in Thee, and in the shadow of Thy wings will I hope, until iniquity pass away.":
        "Orthodox Service Quotes",
    "For the Peace of the world": "Orthodox Service Quotes"

}

In [271]:
scores["Query Category"] = scores["Query"].map(query_categories)
#scores

In [272]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method", "numbered_result", "Similarity Score (%)", "Text", "Psalm Num", 
                  "Verse", "User", "Score" ]]

#scores

In [273]:
# reordering the columns of the dataframe
scores = scores [["Query", "Query Category", "Method","Similarity Score (%)", "numbered_result",
                  "Text", "Psalm Num", "Verse", "User", "Score" ]]

#scores

In [274]:
# filtering to only be studying the top 5 results from each query
scores = scores[scores['numbered_result'] != 6]


pd.set_option("display.max_rows", 50)

#scores[scores['Method'] == 'TFIDF']

In [275]:
external = external[external['numbered_result'] != 6]

---

# Analysis

## Inner-Annotator Agreement
Every person that contributed to the scoring of the results aproached them differently even though the same intetion was behind each score conceived. Each result of the *236 results*, was scored up to three times. This may cause discrepency in how each result was scored overall. **Inner-Anotator Agreement** works at trying to normalize the differeint in scored overall, and for each indivual result. 
There are a few different metricsx that handel this. The data for this study is ordinal which means that a `1` is closer to `2` than `5`, making this just just a category. For this reason **Krippendorff's** alpha agreement is what's going to be used. 

The overall metric consists of the following equation:
$$
\alpha = 1 - \frac{D_0}{D_e}
$$

Where $D_0$ is:
$$
D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2
$$

And $D_e$ is:
$$
D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ ordinal} \, \delta_{ck}^2
$$

In [276]:
import pandas as pd
import numpy as np

Code for: $$D_o = \frac{1}{n} \sum_{c} \sum_{k} o_{ck} \, \delta_{ck}^2$$

In [277]:
import numpy as np

o_ck = [[0.0]*11 for _ in range(11)]
pairable_n = 0

o_ck

[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

Building the $o_ck$ portion of $D_0$:

In [278]:
pairable_n = 0

# coincidence matrix (0–10 scale)
o_ck = [[0.0]*11 for _ in range(11)]

# marginal frequencies n_c
counts = [0]*11

In [279]:
def process_row_external(row):

    scores = [row['Score1'], row['Score2'], row['Score3']]

    valid_scores = []
    for s in scores:
        if not pd.isna(s) and s != "":
            val = int(float(s))
            valid_scores.append(val)
            counts[val] += 1   # <-- THIS is how you get counts

    m_u = len(valid_scores)

    if m_u < 2:
        return

    global pairable_n
    pairable_n += m_u

    for i in range(m_u):
        for j in range(m_u):
            if i == j:
                continue
            c = valid_scores[i]
            k = valid_scores[j]
            o_ck[c][k] += 1/(m_u-1)

In [280]:
for _, row in df.iterrows():
    process_row_external(row)

print(counts)

[49, 74, 55, 48, 38, 50, 54, 44, 74, 63, 63]


<div style=" padding:10px; display:inline-block;">
$$
\delta^2_{ck} = \left( \sum_{g=c}^{\max(c,g)} n_g - \frac{n_c + n_k}{2} \right)^2
$$
</div>

In [281]:
def ordinal_delta_sq(c, k):
    if c == k:
        return 0.0
    
    low = min(c, k)
    high = max(c, k)

    # cumulative counts between ranks
    cumu = sum(counts[g] for g in range(low, high+1))

    # subtract half endpoints
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [282]:
def compute_D_o():
    total_coincidences = sum(sum(row) for row in o_ck)
    if total_coincidences == 0:
        return None  # or 0

    total = 0
    for c in range(len(o_ck)):
        for k in range(len(o_ck)):
            delta_sq = ordinal_delta_sq(c, k)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [283]:
pairable_n = 0
o_ck = [[0.0]*11 for _ in range(11)]
counts = [0]*11  # important if using ordinal_delta_sq

for idx, row in df.iterrows():
    process_row_external(row)

D_o = compute_D_o()

D_o

42920.970637583894

    d_0(temp.iloc[0]
### Building the Coincidences Matrix

In [284]:
o_ck

[[11.0, 8.0, 8.0, 4.0, 1.5, 2.5, 4.5, 3.0, 2.0, 3.5, 0.0],
 [8.0, 11.0, 10.0, 4.0, 4.0, 5.0, 5.0, 6.0, 10.5, 5.0, 4.5],
 [8.0, 10.0, 5.0, 6.0, 3.5, 5.0, 2.5, 4.5, 3.5, 5.0, 2.0],
 [4.0, 4.0, 6.0, 3.0, 4.5, 4.0, 7.5, 2.5, 8.0, 1.5, 3.0],
 [1.5, 4.0, 3.5, 4.5, 1.0, 5.5, 4.0, 1.5, 5.0, 2.0, 1.5],
 [2.5, 5.0, 5.0, 4.0, 5.5, 4.0, 4.5, 4.0, 4.0, 6.0, 3.5],
 [4.5, 5.0, 2.5, 7.5, 4.0, 4.5, 5.0, 5.0, 7.0, 5.0, 3.0],
 [3.0, 6.0, 4.5, 2.5, 1.5, 4.0, 5.0, 6.0, 2.0, 3.5, 6.0],
 [2.0, 10.5, 3.5, 8.0, 5.0, 4.0, 7.0, 2.0, 12.0, 8.5, 9.5],
 [3.5, 5.0, 5.0, 1.5, 2.0, 6.0, 5.0, 3.5, 8.5, 12.0, 9.0],
 [0.0, 4.5, 2.0, 3.0, 1.5, 3.5, 3.0, 6.0, 9.5, 9.0, 18.0]]

### $D_e$
Code for: $D_e = \frac{1}{n(n-1)} \sum_{c} \sum_{n_c} n_c * n_{k\ metric} \, \delta_{ck}^2$

*Where*:
> - $n_c$ = number of times score `c` occurs in the dataset  
> - $n_k$ = number of times score `k` occurs in the dataset  
> - $\delta_{ck}^2$ = squared distance between scores `c` and `k`  
> - `n(n-1)` = total number of pairs in the dataset

Code for 
$$
D_e = \frac{\sum_c \sum_k n_c n_k \left( \sum_{g=\min(c,k)}^{\max(c,k)} n_g - \frac{n_c + n_k}{2} \right)^2}{n(n-1)}, 
\quad n = \sum_c n_c
$$

In [285]:
def compute_D_e():

    n = sum(counts)

    total = 0
    for c in range(11):
        for k in range(11):
            delta_sq = ordinal_delta_sq(c, k)
            total += counts[c] * counts[k] * delta_sq

    return total / (n*(n-1))

Now we can refer to the original formula: $\alpha = 1 - \frac{D_0}{D_e}$

In [286]:
def alpha():
    d_o = compute_D_o()
    d_e = compute_D_e()
    print("D_e = " + str(d_e))
    return 1 - (d_o / d_e)

In [287]:
external.head()

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p06,5
1,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p03,1
2,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,p08,3
3,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p03,1
4,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,p04,8


In [288]:
alpha()

D_e = 61945.78559738134


0.3071204082138833

- `02/20/2026`- $\alpha = 0.614410708025859$
    - I was not properly computing $D_e$ correctly. I was using $(c-k)^2$ rather than $\delta^2_{ck}$. The code above represents the ordinal computations.
- `02/21/2026`- $\alpha = 0.9999895570896626$
Debugging

## Testing with the offical python package


Confirming my results with the offical pacakge before going further. 

In [289]:
# %pip install krippendorff
import krippendorff

In [290]:
import numpy as np
import pandas as pd

temp = df

cols_without = ['Score1', 'Score2', 'Score3']
cols_with = ['CadenScore', 'Score1', 'Score2', 'Score3']

temp[cols_without] = temp[cols_without].apply(pd.to_numeric, errors='coerce')
temp[cols_with] = temp[cols_with].apply(pd.to_numeric, errors='coerce')

In [291]:
data_without_caden = temp[['Score1', 'Score2', 'Score3']].T.to_numpy()

alpha_without = krippendorff.alpha(
    reliability_data=data_without_caden,
    level_of_measurement='ordinal'
)

print("Without Caden:", alpha_without)

Without Caden: 0.30683274368139135


# Rebuilding

Resuilding the metric computation to re compute the score to be able to compute quickly for specfric sets of the data. 

In [292]:
def build_coincidence_matrix(df, score_columns, max_score=10):

    counts = [0]*(max_score+1)
    o_ck = np.zeros((max_score+1, max_score+1))
    pairable_n = 0

    for _, row in df.iterrows():
        scores = [row[col] for col in score_columns]
        
        valid_scores = []
        for s in scores:
            if not pd.isna(s) and s != "":
                val = int(float(s))
                valid_scores.append(val)
                counts[val] += 1

        m_u = len(valid_scores)
        if m_u < 2:
            continue

        pairable_n += m_u

        for i in range(m_u):
            for j in range(m_u):
                if i == j:
                    continue
                c = valid_scores[i]
                k = valid_scores[j]
                o_ck[c][k] += 1/(m_u-1)

    return counts, o_ck, pairable_n

In [293]:
def ordinal_delta_sq(c, k, counts):
    if c == k:
        return 0.0

    low = min(c, k)
    high = max(c, k)

    cumu = sum(counts[g] for g in range(low, high+1))
    cumu -= (counts[c] + counts[k]) / 2

    return cumu ** 2

In [294]:
def compute_D_o(o_ck, counts, max_score=10):

    total_coincidences = o_ck.sum()
    if total_coincidences == 0:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += o_ck[c][k] * delta_sq

    return total / total_coincidences

In [295]:
def compute_D_e(counts, max_score=10):

    n_total = sum(counts)
    if n_total < 2:
        return None

    total = 0
    for c in range(max_score+1):
        for k in range(max_score+1):
            delta_sq = ordinal_delta_sq(c, k, counts)
            total += counts[c] * counts[k] * delta_sq

    return total / (n_total * (n_total - 1))

In [296]:
def compute_alpha(D_o, D_e):

    if D_o is None or D_e is None:
        return None

    if D_e == 0:
        return 1.0

    return 1 - (D_o / D_e)

In [297]:
def krippendorff_alpha_ordinal(df, score_columns, max_score=10):

    counts, o_ck, pairable_n = build_coincidence_matrix(
        df, score_columns, max_score
    )

    D_o = compute_D_o(o_ck, counts, max_score)
    D_e = compute_D_e(counts, max_score)

    alpha = compute_alpha(D_o, D_e)

    return alpha #, D_o, D_e, counts, o_ck

With these functions built, I want to veriofy they are working the way they did before. 

In [298]:
krippendorff_alpha_ordinal(df, ['Score1', 'Score2', 'Score3'], max_score=10)

0.3071204082138833

We are getting the same score a before. This is an easier way of applying the metic because it is easier to change what data to compute. Lets. work on trying different parts of the data. 

### Adding my score

In [299]:
krippendorff_alpha_ordinal(df, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)

0.28217301607656475

### Looking at each indivual Algorithm
#### TFIDF

In [300]:
temp = df[df['Method'] == 'TFIDF']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.3924936002010967
With Caden:  0.2619906930603101


#### TFIDF x GLoVE

In [301]:
temp = df[df['Method'] == 'TFIDF_GLoVe']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.34171761059537276
With Caden:  0.3650689085471893


#### BERT

In [302]:
temp = df[df['Method'] == 'BERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.2448173330170701
With Caden:  0.22447470253447965


#### SBERT

In [303]:
temp = df[df['Method'] == 'SBERT']

print("Witout Caden: ",
      (krippendorff_alpha_ordinal(temp, ['Score1', 'Score2', 'Score3'], max_score=10)))


print("With Caden: ", 
      (krippendorff_alpha_ordinal(temp, ['CadenScore', 'Score1', 'Score2', 'Score3'], max_score=10)))


Witout Caden:  0.18868194484568535
With Caden:  0.20200153863725112


## Investigating the Low Alpha Score

The overall `Krippendorff Aplpha` score I calculated was
$$\alpha_{w/out\ Caden} = 0.3132363233535458$$
and 
$$\alpha_{w/ \ Caden} = 0.27457889178612493$$

These tell us that the scores between everyone is not reliable, Lets see what might be contributing to that. 

### Looking at the everage score of each user

In [324]:
caden

,Query,Method,numbered_result,Similarity Score (%),Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,TFIDF_GLoVe,1,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,TFIDF_GLoVe,2,25.67,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,TFIDF_GLoVe,3,21.90,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,TFIDF_GLoVe,4,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,TFIDF_GLoVe,5,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,2,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",caden,2
232,Verses where the psalmist remembers past deliv...,TFIDF,3,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,caden,1
233,Verses where the psalmist remembers past deliv...,TFIDF,4,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,caden,3
234,Verses where the psalmist remembers past deliv...,TFIDF,5,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",caden,8


In [322]:
import pandas as pd

# Convert 'Score' to numeric, invalid parsing becomes NaN
scores['Score'] = pd.to_numeric(scores['Score'], errors='coerce')

# Optional: drop rows where conversion failed
scores = scores.dropna(subset=['Score'])

# Finally, convert to integer
scores['Score'] = scores['Score'].astype(int)

scores['Score'].count()

778

In [306]:
scores.pivot_table(index='User', values='Score', aggfunc='mean')

,Score
User,
caden,5.986047
p01,5.042553
p02,6.041667
p03,3.388235
p04,5.809524
p05,8.295775
p06,4.404255
p07,6.272727
p08,6.322034


In [307]:
scores['Score'].mean()

5.353470437017995

In [308]:
import numpy as np

def score_category(score):
    # If score is missing or not a number, return np.nan
    if pd.isna(score) or score == '':
        return np.nan
    # Otherwise convert to float and categorize
    score = float(score)
    if score <= 3:
        return 1  # Low
    elif score <= 7:
        return 2  # Medium
    else:
        return 3  # High

In [309]:
df['Score_Caden Category'] = df['CadenScore'].apply(score_category)

df['Score_1 Category'] = df['Score1'].apply(score_category)
df['Score_2 Category'] = df['Score2'].apply(score_category)
df['Score_3 Category'] = df['Score3'].apply(score_category)

In [310]:
df.head()

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,Score_Caden Category,Score_1 Category,Score_2 Category,Score_3 Category
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,5.0,p06,1.0,p03,3.0,p08,3,2.0,1.0,1.0
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,1.0,p03,8.0,p04,7.0,p06,2,1.0,3.0,2.0
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,7.0,p06,1.0,p03,3.0,p10,1,2.0,1.0,1.0
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,7.0,p06,6.0,p05,5.0,p09,3,2.0,2.0,2.0
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,2.0,p01,0.0,p03,2.0,p08,2,1.0,1.0,1.0


### Using Krippendorff ALpha for categorical scores

In [311]:
print(df.columns.tolist())

['Query', 'Method', 'Similarity Score (%)', 'Text', 'Psalm Num', 'Verse', 'CadenScore', 'Score1', 'User1', 'Score2', 'User2', 'Score3', 'User3', 'Score_Caden Category', 'Score_1 Category', 'Score_2 Category', 'Score_3 Category']


In [312]:
df

,Query,Method,Similarity Score (%),Text,Psalm Num,Verse,CadenScore,Score1,User1,Score2,User2,Score3,User3,Score_Caden Category,Score_1 Category,Score_2 Category,Score_3 Category
0,Create in me a clean heart,TFIDF_GLoVe,26.10,Psalter,100,I will sing of mercy and judgment unto Thee O ...,9,5.0,p06,1.0,p03,3.0,p08,3,2.0,1.0,1.0
1,Create in me a clean heart,TFIDF_GLoVe,25.67,Bible,4,For the End in psalms an ode by David You hear...,6,1.0,p03,8.0,p04,7.0,p06,2,1.0,3.0,2.0
2,Create in me a clean heart,TFIDF_GLoVe,21.90,Bible,31,By David concerning understanding Blessed are ...,3,7.0,p06,1.0,p03,3.0,p10,1,2.0,1.0,1.0
3,Create in me a clean heart,TFIDF_GLoVe,18.33,Psalter,31,Blessed are they whose iniquities are forgiven...,10,7.0,p06,6.0,p05,5.0,p09,3,2.0,2.0,2.0
4,Create in me a clean heart,TFIDF_GLoVe,18.12,Bible,61,For the End for Jeduthun a psalm by David Shal...,7,2.0,p01,0.0,p03,2.0,p08,2,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,Verses where the psalmist remembers past deliv...,TFIDF,9.98,Psalter,23,"The earth is the Lord's, and the fulness there...",2,0.0,p01,0.0,p06,7.0,p04,1,1.0,1.0,2.0
232,Verses where the psalmist remembers past deliv...,TFIDF,7.79,Bible,130,1An ode of ascents by David OLord My heart is ...,1,0.0,p03,0.0,p17,NaN,,1,1.0,1.0,NaN
233,Verses where the psalmist remembers past deliv...,TFIDF,7.08,Psalter,61,Shall not my soul be subject unto God? for fro...,3,10.0,p06,NaN,,NaN,,1,3.0,NaN,NaN
234,Verses where the psalmist remembers past deliv...,TFIDF,5.61,Psalter,35,"The transgressor, that he may sin, saith withi...",8,2.0,p10,7.0,p05,NaN,,3,1.0,2.0,NaN


In [313]:
# Step 2: Convert to numpy array
data_without_caden = df[['Score_1 Category','Score_2 Category','Score_3 Category']].T.to_numpy()

# Step 3: Compute Krippendorff (with missing values preserved)
alpha_without = krippendorff.alpha(
    reliability_data=data_without_caden,
    level_of_measurement='ordinal'
)

print("Without Caden:", alpha_without)

Without Caden: 0.2577499693585348


From a different studying Using `Krippendorff's alpha`: 

**"A Validated Scoring Rubric for Explain-in-Plain-English Questions" <br>**
> "we first applied z-score standardization on a per question basis to account for the variations in question difficulty. After standardization, we computed the average z-score for each student to account for the fact that some students answered fewer code reading questions. "
>
- There is variation in every rater's view of the Psalms in addition to the different interpretations of each query
- No one was given an example of this because it up to interpretation as everyone is seeking the Psalsm with different prespectives, as I make note of within the introduction of the Poster and Paper. 
- Applying the `Z-score` standarization may help in these realistic short comings. 

$$
z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

$$
\mu_j = \frac{1}{n} \sum_{i=1}^{n} x_{ij}
$$

$$
\sigma_j = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_{ij} - \mu_j)^2}
$$

After the z-score is implented the data turns into interval data

In [314]:
df_z

,CadenScore,Score1,Score2,Score3
0,0.967826,-0.032522,-1.276484,-0.610786
1,-0.008272,-1.279194,0.799708,0.603558
2,-0.984370,0.590814,-1.276484,-0.610786
3,1.293193,0.590814,0.206510,-0.003614
4,0.317094,-0.967526,-1.573082,-0.914373
...,...,...,...,...
231,-1.309737,-1.590862,-1.573082,0.603558
232,-1.635103,-1.590862,-1.573082,NaN
233,-0.984370,1.525818,NaN,NaN
234,0.642460,-0.967526,0.503109,NaN


In [315]:
# Keep only numeric scores
raters = ['CadenScore', 'Score1', 'Score2', 'Score3']

# Z-score standardization (per rater)
df_z = (df[raters] - df[raters].mean()) / df[raters].std()

# Convert to numpy for Krippendorff
import krippendorff
data = df_z.T.to_numpy()

alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='interval')

print("Krippendorff alpha (z-score standardized):", alpha)

Krippendorff alpha (z-score standardized): 0.28024168485584044


In [316]:
df_adjusted

,CadenScore,Score1,Score2,Score3
0,2.974576,-0.104348,-4.303738,-2.011905
1,-0.025424,-4.104348,2.696262,1.988095
2,-3.025424,1.895652,-4.303738,-2.011905
3,3.974576,1.895652,0.696262,-0.011905
4,0.974576,-3.104348,-5.303738,-3.011905
...,...,...,...,...
231,-4.025424,-5.104348,-5.303738,1.988095
232,-5.025424,-5.104348,-5.303738,NaN
233,-3.025424,4.895652,NaN,NaN
234,1.974576,-3.104348,1.696262,NaN


Krippendorff’s alpha (ordinal) looks at how raters rank or differentiate items relative to each other.

If raters disagree on the ordering of items, subtracting each rater’s mean doesn’t help — it only shifts the scores up or down.

---

In [317]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

temp = df[df['Method'] == 'TFIDF_GLoVe']

def weighted_kappa_average(dataframe, columns):

    kappas = []

    for i in range(len(columns)):
        for j in range(i+1, len(columns)):

            r1 = dataframe[columns[i]]
            r2 = dataframe[columns[j]]

            # Remove missing values
            valid = ~(r1.isna() | r2.isna())

            kappa = cohen_kappa_score(
                r1[valid],
                r2[valid],
                weights='quadratic'
            )

            kappas.append(kappa)

    return np.mean(kappas)

In [318]:
print("Weighted Kappa Without Caden:",
      weighted_kappa_average(
          df,
          ['Score1','Score2','Score3']
      ))

Weighted Kappa Without Caden: 0.319204255931687


In [319]:
scores.head()

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7


In [320]:
pd.pivot_table(data=scores, index=['User', "Method"], values='Score', aggfunc = 'count')

Score
User  Method            
caden BERT            55
      SBERT           55
      TFIDF           50
      TFIDF_GLoVe     55
p01   BERT            28
      SBERT           22
      TFIDF           22
      TFIDF_GLoVe     22
p02   BERT             6
      SBERT            7
      TFIDF            7
      TFIDF_GLoVe      4
p03   BERT            19
      SBERT           24
      TFIDF           19
      TFIDF_GLoVe     23
p04   BERT             5
      SBERT            7
      TFIDF            2
      TFIDF_GLoVe      7
p05   BERT            25
      SBERT           14
      TFIDF           17
      TFIDF_GLoVe     15
p06   BERT            23
      SBERT           23
      TFIDF           17
      TFIDF_GLoVe     31
p07   BERT             4
      SBERT            8
      TFIDF            4
      TFIDF_GLoVe      6
p08   BERT            13
      SBERT           12
      TFIDF           16
      TFIDF_GLoVe     18
p09   BERT             9
      SBERT            5
      TFIDF            8
      TFIDF_GLoVe      4
p10   BERT            14
      SBERT           17
      TFIDF           15
      TFIDF_GLoVe     11
p13   BERT             2
      SBERT            2
      TFIDF            2
p17   TFIDF            1
      TFIDF_GLoVe      3

In [321]:
scores

,Query,Query Category,Method,Similarity Score (%),numbered_result,Text,Psalm Num,Verse,User,Score
0,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,26.10,1,Psalter,100,I will sing of mercy and judgment unto Thee O ...,caden,9
1,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,25.67,2,Bible,4,For the End in psalms an ode by David You hear...,caden,6
2,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,21.90,3,Bible,31,By David concerning understanding Blessed are ...,caden,3
3,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.33,4,Psalter,31,Blessed are they whose iniquities are forgiven...,caden,10
4,Create in me a clean heart,Phrase/Exact Match Queries,TFIDF_GLoVe,18.12,5,Bible,61,For the End for Jeduthun a psalm by David Shal...,caden,7
...,...,...,...,...,...,...,...,...,...,...
932,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.79,3,Bible,130,1An ode of ascents by David OLord My heart is ...,p03,0
933,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.79,3,Bible,130,1An ode of ascents by David OLord My heart is ...,p17,0
935,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,7.08,4,Psalter,61,Shall not my soul be subject unto God? for fro...,p06,10
938,Verses where the psalmist remembers past deliv...,Long/Complex Queries,TFIDF,5.61,5,Psalter,35,"The transgressor, that he may sin, saith withi...",p10,2
